# 06 · Validación en el Clausura 2026

**Qué hace.** Abre por primera y única vez el torneo de prueba (19 partidos de Jardine en el CL26) y responde si cada principio **mantiene su dirección** frente a Solari y Ortiz en partidos que el análisis no vio. Revisa también el índice y la matriz de situaciones. Es la evaluación final del plan (`contexto/plan_cierre.md`, fase A).

**Veredicto** (`ojo.lentes.validacion`):
- *se sostiene*: misma dirección que en entrenamiento y el IC no cruza 0;
- *misma dirección, sin certeza*;
- *no se sostiene*;
- *sin muestra suficiente*: menos de `validacion.min_partidos_prueba` partidos con el quiebre.

> IMPOR1: los principios se fijaron el 18 oct con el criterio v2 (notebooks 02–04) **antes** de abrir este torneo y no se cambian después de ver estos resultados. Se informa el veredicto aunque sea negativo.
>
> IMPOR2: solo agregados; el notebook se sube sin salidas.

**Qué necesita.** Pasos 02–05 corridos, con los partidos del CL26 incluidos en `outputs/panel/`.

**Cómo probarlo.** En VS Code, kernel `.venv`, *Run All*. Tarda 1–2 minutos. Resultados en `outputs/hallazgos/validacion_cl26.parquet` y `matriz_cl26.parquet`.

In [ ]:
import numpy as np
import pandas as pd

from ojo.config import DIR_SALIDAS, params
from ojo.datos import almacen
from ojo.framework import juego
from ojo.framework import partidos as fp
from ojo.lentes import defensa, efectos, hallazgo, indice, jugadores, minuto, situaciones, validacion

P = params()
EQUIPO = P["equipo"]["team_id"]
RNG = np.random.default_rng(P["simulacion"]["semilla"])
DIR_HALLAZGOS = DIR_SALIDAS / "hallazgos"
ENTRENADORES = {d["manager_id"]: d["nombre"].split()[-1] for d in P["predecesores"]}
ENTRENADORES[P["entrenador"]["manager_id"]] = P["entrenador"]["nombre"].split()[-1]
NUEVO, PREVIOS, PRUEBA = list(ENTRENADORES.values())[-1], "Predecesores", "Prueba (CL26)"

partidos = fp.con_torneo(fp.del_equipo(fp.validos(almacen.cargar("partidos")), EQUIPO))
partidos = partidos[partidos["entrenador_id"].isin(ENTRENADORES)].copy()
partidos["prueba"] = fp.es_prueba(partidos)
partidos["grupo"] = np.where(partidos["prueba"], PRUEBA, np.where(partidos["entrenador_id"].map(ENTRENADORES) == NUEVO, NUEVO, PREVIOS))
ids = partidos["match_id"].astype(int).tolist()
eventos = almacen.cargar("eventos", ids, columnas=sorted(set(efectos.COLUMNAS_EVENTOS) | {
    "x", "y", "fin_x", "fin_y", "patron", "posicion_id", "pase_tipo", "duelo_tipo", "recuperacion_fallida",
    "intercepcion_resultado", "duracion", "es_transicion", "directness"}))
panel = minuto.cargar_panel("minuto").query("match_id in @ids")
aristas = minuto.cargar_panel("aristas").query("match_id in @ids")
partidos.groupby("grupo").agg(partidos=("match_id", "size"), torneos=("torneo_corto", lambda s: ", ".join(s.unique())))

## 1. Principios que se miden por partido

*Sale jugando corto* (largo medio de sus pases en juego abierto) y *refresca las bandas* (cambios tácticos de defensas por partido). Diferencia frente a Solari y Ortiz en entrenamiento y en el CL26.

In [ ]:
pases = eventos[(eventos["equipo_id"] == EQUIPO) & (eventos["tipo"] == "Pass") & juego.es_juego_abierto(eventos)]
largo = np.hypot(pases["fin_x"] - pases["x"], pases["fin_y"] - pases["y"]).groupby(pases["match_id"]).mean()
subs = jugadores.sustituciones(eventos, EQUIPO)
defensas = subs[~subs["lesion"] & (subs["linea"] == "Defensa")].groupby("match_id").size()
por_partido = partidos[["match_id", "grupo"]].assign(
    largo_pase=lambda d: d["match_id"].map(largo), tacticos_defensa=lambda d: d["match_id"].map(defensas).fillna(0))

filas = []
for titulo, col in (("Sale jugando corto", "largo_pase"), ("Refresca las bandas", "tacticos_defensa")):
    g = {k: por_partido[por_partido["grupo"] == k] for k in (NUEVO, PREVIOS, PRUEBA)}
    entr = validacion.diferencia(g[NUEVO], g[PREVIOS], col, RNG)
    prue = validacion.diferencia(g[PRUEBA], g[PREVIOS], col, RNG)
    filas.append({"principio": titulo, "medida": col, "dif_entrenamiento": entr["dif"], "dif_prueba": prue["dif"],
                  "bajo": prue["bajo"], "alto": prue["alto"], "partidos_prueba": prue["partidos"], "eventos_prueba": prue["eventos"],
                  "veredicto": validacion.veredicto(entr["dif"], prue["dif"], prue["bajo"], prue["alto"], prue["partidos"])})
pd.DataFrame(filas).round(3)

## 2. Principios de quiebre

El mismo diseño del notebook 02 (`ojo.lentes.efectos`): cada quiebre del CL26 frente a ventanas de control del mismo torneo. La diferencia se toma frente a los efectos de Solari y Ortiz.

In [ ]:
e = efectos.efectos(eventos, panel, aristas, partidos, EQUIPO)
PRINCIPIOS_QUIEBRE = [
    ("Con uno más, hunde al rival", "expulsion_favor", "rival", "altura_sin_balon"),
    ("Una lesión no lo frena", "sustitucion_lesion_propia", "propio", "presiones"),
    ("Tras recibir gol, el rival no lo aprieta", "gol_contra", "rival", "contrapresiones"),
    ("Alerta: cambiar el sistema le cuesta", "cambio_sistema_propio", "propio", "xg"),
]
for titulo, tipo, lado, metrica in PRINCIPIOS_QUIEBRE:
    sub = e[(e["tipo"] == tipo) & (e["lado"] == lado) & (e["metrica"] == metrica)].dropna(subset=["efecto"])
    g = {k: sub[sub["grupo"] == k] for k in (NUEVO, PREVIOS, PRUEBA)}
    entr = validacion.diferencia(g[NUEVO], g[PREVIOS], "efecto", RNG)
    prue = validacion.diferencia(g[PRUEBA], g[PREVIOS], "efecto", RNG)
    filas.append({"principio": titulo, "medida": f"{tipo} · {lado} · {metrica}", "dif_entrenamiento": entr["dif"],
                  "dif_prueba": prue["dif"], "bajo": prue["bajo"], "alto": prue["alto"], "partidos_prueba": prue["partidos"],
                  "eventos_prueba": prue["eventos"],
                  "veredicto": validacion.veredicto(entr["dif"], prue["dif"], prue["bajo"], prue["alto"], prue["partidos"])})
resultado = pd.DataFrame(filas)
resultado["veredicto_texto"] = resultado["veredicto"].map(validacion.VEREDICTOS)
resultado.to_parquet(DIR_HALLAZGOS / "validacion_cl26.parquet", index=False)
resultado.round(3)

## 3. Índice y matriz en el CL26

El índice del América con la escala de los minutos de entrenamiento, y la matriz de situaciones del CL26 frente a la de AP23–AP25 (notebook 05).

In [ ]:
comp = indice.componentes(panel, indice.piezas_por_minuto(eventos))
entrenamiento = comp[comp["match_id"].isin(partidos.loc[~partidos["prueba"], "match_id"])]
tab = indice.calcular(comp, indice.ajustar(entrenamiento)).merge(partidos[["match_id", "grupo"]], on="match_id")
ind = tab.groupby(["grupo", "lado"])["indice"].mean().unstack().round(1)

CLAVES = ["match_id", "periodo", "minuto"]
lados = panel.pivot_table(index=CLAVES, columns="lado", values=["xg", "pases_ultimo_tercio"], aggfunc="sum", fill_value=0)
t = pd.DataFrame({"xg_favor": lados[("xg", minuto.PROPIO)], "xg_contra": lados[("xg", minuto.RIVAL)],
                  "tercio_favor": lados[("pases_ultimo_tercio", minuto.PROPIO)],
                  "tercio_contra": lados[("pases_ultimo_tercio", minuto.RIVAL)]})
d = defensa.piezas_por_minuto(eventos)
propio, rival = d[d["equipo_id"] == EQUIPO].set_index(CLAVES), d[d["equipo_id"] != EQUIPO].set_index(CLAVES)
t = t.join(pd.DataFrame({"acciones_def": propio["acciones_def_zona"], "rec_n": propio["recuperaciones_n"],
                         "rec_x": propio["recuperaciones_x_suma"], "pases_rival_zona": rival["pases_zona_propia"]}), how="outer")
t[situaciones.SUMAS] = t[situaciones.SUMAS].fillna(0.0)
t = t.join(tab.pivot_table(index=CLAVES, columns="lado", values="indice")
           .rename(columns={minuto.PROPIO: "indice_favor", minuto.RIVAL: "indice_rival"}))
t = t.join(situaciones.por_minuto(eventos, EQUIPO).set_index(CLAVES)).dropna(subset=["dif_goles"]).reset_index()
t["situacion"] = situaciones.asignar(t)
t = t.merge(partidos[["match_id", "grupo"]], on="match_id")
t["xg_dif"] = t["xg_favor"] - t["xg_contra"]
cl26 = t[t["grupo"] == PRUEBA]
filas_m = []
for sit, nombre in situaciones.SITUACIONES.items():
    sub = cl26[cl26["situacion"] == sit]
    if sub.empty:
        continue
    r = situaciones.resumir(sub, ["situacion"]).iloc[0].to_dict()
    bajo, alto = hallazgo.intervalo(hallazgo.medias_bootstrap(sub, "xg_dif", RNG))
    r.update({"nombre": nombre, "xg_dif_bajo": 90 * bajo, "xg_dif_alto": 90 * alto})
    filas_m.append(r)
matriz_cl26 = pd.DataFrame(filas_m)
entr = pd.read_parquet(DIR_HALLAZGOS / "matriz.parquet")[["situacion", "xg_dif_90", "xg_dif_bajo", "xg_dif_alto", "indice"]]
matriz_cl26 = matriz_cl26.merge(entr.add_suffix("_entrenamiento").rename(columns={"situacion_entrenamiento": "situacion"}),
                                on="situacion", how="left")
matriz_cl26.to_parquet(DIR_HALLAZGOS / "matriz_cl26.parquet", index=False)
print(ind)
matriz_cl26[["nombre", "minutos", "partidos", "indice", "indice_entrenamiento", "xg_dif_90", "xg_dif_bajo", "xg_dif_alto",
             "xg_dif_90_entrenamiento"]].round(2)

## Lectura

Corrida del 18 oct: primera y única apertura del CL26 (19 partidos de Jardine, incluidos los cuartos de final contra Pumas). Los principios se habían fijado antes con el criterio v2.

1. **Ningún principio se invierte.**
    - **Se sostienen dos:**
        - *Sale jugando corto*: −1.40 yd frente a Solari y Ortiz (IC −2.02 a −0.79; en entrenamiento, −1.51).
        - La *alerta del cambio de sistema*: −0.028 xG por minuto en los 10 minutos siguientes (IC −0.043 a −0.014). Son solo 4 partidos con cambio de sistema.
    - **Mantienen la dirección sin certeza** (muestra chica):
        - *refresca las bandas*: +0.31 cambios tácticos de defensas por partido; en entrenamiento, +0.73;
        - *una lesión no lo frena*: 7 partidos;
        - *tras recibir gol, el rival no lo aprieta*: 7 partidos.
    - *Con uno más, hunde al rival* queda **sin muestra suficiente**: las expulsiones a favor del CL26 no dejan ventanas válidas.
2. **El índice confirma la identidad.** En el CL26 el América promedia 62 (en AP23–AP25, 57) y sus rivales 48, igual que antes. El equipo de Jardine sigue siendo el más proactivo de la cancha.
3. **La matriz repite su advertencia principal.**
    - **Ganando por 1** el América vuelve a replegarse (índice 46) y el xG neto cae a −0.14 por 90 (IC −0.85 a 0.59). Es el mismo patrón de entrenamiento (49 y +0.31) en el torneo que no se vio, aunque con un IC muy ancho; la hipótesis *sostener la iniciativa con 1-0* sigue siendo hipótesis (auditoría del 19 oct, N1).
    - **Empatando en el primer tiempo** propone igual (índice 65), pero ya no saca ventaja (xG neto −0.03, frente a +0.50 en entrenamiento). Es una señal para revisar, con muestra de un torneo.
4. **Cautelas.** Un torneo de 19 partidos da intervalos anchos: "misma dirección, sin certeza" no confirma ni descarta. Lo que se presenta como validado son las dos filas que se sostienen y la dirección de las demás.